# Durable execution

**Durable execution** is a technique in which a process or workflow saves its progress at key points, allowing it to pause and later resume exactly where it left off. This is particularly useful in scenarios that require human-in-the-loop, where users can inspect, validate, or modify the process before continuing, and in long-running tasks that might encounter interruptions or errors (e.g., calls to an LLM timing out). By preserving completed work, durable execution enables a process to resume without reprocessing previous steps — even after a significant delay (e.g., a week later).


LangGraph’s built-in persistence layer provides durable execution for workflows, ensuring that the state of each execution step is saved to a durable store. This capability guarantees that if a workflow is interrupted — whether by a system failure or for human-in-the-loop interactions — it can be resumed from its last recorded state.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

## 1. What is Durable Execution (Simple Idea)

Durable execution means:

A workflow saves its progress so it can **resume later from where it stopped**.

Example real life:

Imagine a **food delivery workflow**

1. Order received

2. Restaurant prepares food

3. Delivery boy picks order

4. Delivery completed

Now suppose the server crashes during step 3.

Without durable execution:

* The system **starts again from step 1**

With durable execution:

It **continues from step 3**

LangGraph does the same for **LLM agents and workflows**.

## 2. Why Durable Execution is Important

AI agents often run long workflows, for example:

* Multi-step reasoning

* API calls

* Database updates

* Human approval steps

* Tool executions

Problems that can happen:

* API timeout

* LLM failure

* Server crash

* Human approval delay

* Network issue

Durable execution ensures:

* The workflow **does not restart**

* It **continues from last checkpoint**.

## 3. Example Without Durable Execution

Example workflow:

```python
User Request
     ↓
Plan Task
     ↓
Call API
     ↓
Save Result
     ↓
Send Response
```

Suppose failure happens at **Call API**.

Without durable execution:

```python
Workflow restarts from beginning
User Request → Plan Task → Call API → ...
```

Everything runs again.

## 4. Example With Durable Execution

LangGraph saves state checkpoints.

```python
Checkpoint 1 → Plan Task
Checkpoint 2 → Call API
Checkpoint 3 → Save Result
```

If system crashes after checkpoint 2:

Resume:

```python
Checkpoint 2 → Save Result → Send Response
```

No repeated work.

## 5. How LangGraph Enables Durable Execution

Durable execution works automatically if you use:

1️⃣ Checkpointer

Example:

```python
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
```

This saves graph state.

2️⃣ Thread ID

Every workflow run needs a thread_id.

Example:

```python
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}
```

This allows LangGraph to:

track execution

resume later

3️⃣ Persistence

When graph runs:

```python
Node execution
↓
State saved
↓
Next node
```

LangGraph stores progress.

In [10]:
from typing import TypedDict


class State(TypedDict):
    text: str
    result: str


def process_text(state: State):
    return {"result": state["text"].upper()}


from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

checkpointer = InMemorySaver()

builder = StateGraph(State)
builder.add_node("process_text", process_text)
builder.add_edge(START, "process_text")
builder.add_edge("process_text", END)

graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "1"}}

graph.invoke({"text": "hello world"}, config=config)
graph.invoke({"text": "goodbye world"}, config=config)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	process_text(process_text)
	__end__([<p>__end__</p>]):::last
	__start__ --> process_text;
	process_text --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. Determinism (Very Important Concept)

Durable execution replays execution.

When resuming:

LangGraph **re-executes previous steps** to reach the stopping point.

Because of this:

Your workflow must be **deterministic**.

Meaning:

Same input → same output.

Bad Example (Non-Deterministic)

```python
import random

def node(state):
    return {"number": random.randint(1,100)}
```

Problem:

When replay happens:

First run → 25
Replay → 80

Workflow becomes inconsistent.

## 8. Side Effects Problem

Side effects are operations like:

* API calls

* database writes

* file writes

* sending emails

Example:

```python
def send_email():
    send_email_to_user()
```

If workflow replays:

Email may be sent **multiple times**.

## 9. Solution → Use Tasks

LangGraph recommends wrapping such operations inside **tasks**.

Tasks store results so they are not **re-executed on replay**.

Example concept:

```python
Task execution
↓
Result stored
↓
Replay uses stored result
```

## 10. Durability Modes

LangGraph provides 3 durability levels.

1️⃣ exit (fastest)

State saved only when graph finishes.

```python
Start
 ↓
Run graph
 ↓
Save state
```

Pros

* fastest

Cons

* crash during execution → progress lost

2️⃣ async (balanced)

State saved asynchronously while next step runs.

Pros

* good performance

* some durability

Cons

* small chance checkpoint not saved

3️⃣ sync (safest)

State saved before every step.

```python
Node execution
↓
Save checkpoint
↓
Next node
```

Pros

* safest

Cons

* slower

Example:

```python
graph.stream(
    input,
    {"durability": "sync"}
)
```

11. Resuming Workflows

Durable execution allows workflows to resume in cases like:

Case 1 — Human in the loop

Example:

AI writes draft
↓
Human reviews
↓
Continue workflow
Case 2 — Failure recovery

Example:

LLM API fails
↓
System restarts
↓
Workflow resumes automatically
Case 3 — Pause and Resume

You can interrupt workflow.

Then continue later.

12. Where Workflow Resumes From

Depends on architecture.

Graph API

Resumes from beginning of node where it stopped.

Example:

Node A
Node B  ← stopped here
Node C

Resume from:

Node B
Subgraph

If subgraph stopped:

Parent Node
  ↓
Subgraph Node B ← stop

Resume from:

Parent node

13. Real Production Example

Typical LangGraph architecture:

User request
     ↓
Planning node
     ↓
Tool execution
     ↓
Database write
     ↓
Human approval
     ↓
Final result

Durable execution ensures:

if server restarts

if LLM fails

if human delays

Workflow continues safely.

14. When Durable Execution is Used

Most useful for:

AI Agents

Example:

Plan
Reason
Use tools
Analyze results
Repeat
Long Running Tasks

Example:

research agents

report generators

data pipelines

Human-in-the-loop

Example:

AI generates contract
↓
Lawyer reviews
↓
AI continues
15. Important Rules (Production)

Follow these rules when using durable execution:

Rule 1

Always use checkpointer

InMemorySaver
PostgresSaver
RedisSaver
Rule 2

Always pass thread_id

configurable.thread_id
Rule 3

Avoid non-deterministic code.

No:

random
time.now()
uuid

unless wrapped.

Rule 4

Wrap side effects in tasks

Examples:

API calls

database writes

email sending

16. One Line Summary

Durable execution =

Persistent + Replayable + Recoverable workflows

Meaning:

Your AI agents never lose progress.